# 02 - Log Parsing and Pattern Extraction

Notebook nay tap trung vao logs de tra loi `WHERE` va `WHAT`.

Cong nghe/method da chot:
- log template mining voi `Drain3` neu co san
- fallback regex normalization de notebook van runnable
- logs duoc dung nhu lop phat hien som nhat cho JVM stress va cache failure


In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

try:
    from drain3 import TemplateMiner
    from drain3.template_miner_config import TemplateMinerConfig
except ImportError:
    TemplateMiner = None
    TemplateMinerConfig = None

BASE = Path(r'D:\AWS\AIOPS-study\g2-data\g2\logs')
METRICS_BASE = Path(r'D:\AWS\AIOPS-study\g2-data\g2\metrics')
ART = Path(r'D:\AWS\AIOPS\w1\lab\artifacts')
ART.mkdir(parents=True, exist_ok=True)
ALERT_TIME = pd.Timestamp('2026-06-01T23:04:00Z')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fbfcfe',
    'grid.alpha': 0.35,
    'axes.titleweight': 'bold',
})


def fmt_time_axis(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H'))


In [ ]:
def normalize(msg: str) -> str:
    msg = re.sub(r'\b[0-9a-f]{16,}\b', '<hex>', msg, flags=re.I)
    msg = re.sub(r'\bORD-[A-Z0-9]+\b', 'ORD-<id>', msg)
    msg = re.sub(r'\b\d+\.\d+\b', '<num>', msg)
    msg = re.sub(r'\b\d+\b', '<num>', msg)
    return msg


def build_template_miner():
    if TemplateMiner is None:
        return None
    cfg = TemplateMinerConfig()
    cfg.load_default_config()
    cfg.profiling_enabled = False
    return TemplateMiner(config=cfg)


def load_log(path: Path):
    miner = build_template_miner()
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            rec['timestamp'] = pd.to_datetime(rec['timestamp'], utc=True)
            rec['template'] = miner.add_log_message(rec['message'])['template_mined'] if miner is not None else normalize(rec['message'])
            rows.append(rec)
    return pd.DataFrame(rows)

cart = load_log(BASE / 'cart-service.log.jsonl')
order = load_log(BASE / 'order-service.log.jsonl')
cart_metrics = pd.read_csv(METRICS_BASE / 'cart-service.csv', parse_dates=['timestamp']).ffill().fillna(0)
cart_metrics['memory_utilization_pct'] = cart_metrics['memory_usage_bytes'] / cart_metrics['memory_limit_bytes'] * 100
restart_events = cart_metrics.loc[cart_metrics['container_restart_count'].diff().fillna(0) > 0, ['timestamp', 'container_restart_count']]
print('cart logs', len(cart), 'order logs', len(order))


In [ ]:
cart_counter = Counter(cart['template'])
cart_top20 = pd.DataFrame(cart_counter.most_common(20), columns=['template', 'count'])

key_rows = []
for signal in [
    'GC overhead limit warning',
    'ProductCatalogCache eviction failed',
    'OutOfMemoryError imminent',
    'Container OOMKilled',
    'Application starting up',
    'Connection pool nearing limit',
]:
    mask = cart['message'].str.contains(signal, case=False, regex=False)
    key_rows.append({'signal': signal, 'first_seen': cart.loc[mask, 'timestamp'].min(), 'count': int(mask.sum())})
key_df = pd.DataFrame(key_rows).sort_values('first_seen')
display(key_df)
display(cart_top20.head(12))


In [ ]:
cart['window_30m'] = cart['timestamp'].dt.floor('30min')
cart['is_warn_error'] = cart['level'].isin(['WARN', 'ERROR', 'FATAL'])
cart_window = cart.groupby('window_30m')['is_warn_error'].sum().reset_index(name='warn_error_count')
log_baseline = cart_window[cart_window['window_30m'] < pd.Timestamp('2026-06-01T12:00:00Z')]['warn_error_count'].median()
cart_window['threshold_2x'] = log_baseline * 2
first_above_2x = cart_window.loc[cart_window['warn_error_count'] > cart_window['threshold_2x'], 'window_30m'].min()
print('30m WARN+ERROR baseline =', log_baseline)
print('first 30m window > 2x baseline =', first_above_2x)

order['window_30m'] = order['timestamp'].dt.floor('30min')
order['is_timeout_refused'] = order['message'].str.contains('timeout|refused|5xx', case=False, regex=True)
order_window = order.groupby('window_30m')['is_timeout_refused'].sum().reset_index(name='timeout_refused_count')
order_baseline = order_window[order_window['window_30m'] < pd.Timestamp('2026-06-01T12:00:00Z')]['timeout_refused_count'].median()
order_first = order_window.loc[order_window['timeout_refused_count'] > order_baseline * 2, 'window_30m'].min()
print('order first elevated 30m window =', order_first)


In [ ]:
# Chart 2: professional 30m log error bars
colors = []
for ts in cart_window['window_30m']:
    if ts < pd.Timestamp('2026-06-01T09:01:00Z'):
        colors.append('#7fb069')
    elif ts < ALERT_TIME:
        colors.append('#f4a261')
    else:
        colors.append('#d62828')

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(cart_window['window_30m'], cart_window['warn_error_count'], width=0.018, color=colors, edgecolor='white')
ax.axhline(log_baseline * 2, color='#264653', linestyle='--', linewidth=1.3, label='2x baseline')
ax.axvline(ALERT_TIME, color='#c1121f', linestyle='--', linewidth=1.4, label='Alert fired')
if pd.notna(first_above_2x):
    first_value = cart_window.loc[cart_window['window_30m'] == first_above_2x, 'warn_error_count'].iloc[0]
    ax.annotate(f'First > 2x baseline\n{first_above_2x:%H:%M}', xy=(first_above_2x, first_value), xytext=(first_above_2x, first_value + 120), arrowprops={'arrowstyle': '->', 'color': '#264653'}, color='#264653', ha='center')
ax.set_title('cart-service WARN + ERROR + FATAL per 30-minute window')
ax.set_xlabel('Timestamp (UTC)')
ax.set_ylabel('Event count')
ax.legend(loc='upper left')
fmt_time_axis(ax)
fig.tight_layout()
fig.savefig(ART / 'chart_02_log_error_rate.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Chart 3: memory with thresholds, OOM, restart, and first root-cause markers
fig, ax1 = plt.subplots(figsize=(16, 7))
ax2 = ax1.twinx()
mem_gb = cart_metrics['memory_usage_bytes'] / (1024**3)
limit_gb = cart_metrics['memory_limit_bytes'].iloc[0] / (1024**3)
threshold_60 = limit_gb * 0.60

ax1.plot(cart_metrics['timestamp'], mem_gb, color='#1d6996', linewidth=1.8, label='Memory usage (GB)')
ax1.axhline(threshold_60, color='#f4a261', linestyle='--', linewidth=1.4, label='60% threshold')
ax1.axhline(limit_gb, color='#c1121f', linestyle='--', linewidth=1.4, label='OOM limit (2GB)')
ax2.step(cart_metrics['timestamp'], cart_metrics['container_restart_count'], where='post', color='#2a9d8f', linewidth=1.8, label='Restart count')

for ts in restart_events['timestamp']:
    ax1.axvline(ts, color='#2a9d8f', linestyle=':', linewidth=1.0, alpha=0.85)

markers = [
    (pd.Timestamp('2026-06-01T06:30:19Z'), 'GC warning', '#f77f00'),
    (pd.Timestamp('2026-06-01T06:32:33Z'), 'Cache eviction failed', '#7b2cbf'),
    (pd.Timestamp('2026-06-01T19:59:02Z'), 'OOMKilled #1', '#c1121f'),
    (ALERT_TIME, 'Alert fired', 'black'),
]
for ts, label, color in markers:
    ax1.axvline(ts, color=color, linestyle='--', linewidth=1.3)

ax1.annotate('Memory > 60%\n18:22', xy=(pd.Timestamp('2026-06-01T18:22:00Z'), threshold_60), xytext=(pd.Timestamp('2026-06-01T17:20:00Z'), threshold_60 + 0.18), arrowprops={'arrowstyle': '->', 'color': '#f4a261'}, color='#bc6c25')
ax1.annotate('OOMKill #1\n19:59', xy=(pd.Timestamp('2026-06-01T19:59:02Z'), 1.55), xytext=(pd.Timestamp('2026-06-01T18:45:00Z'), 1.78), arrowprops={'arrowstyle': '->', 'color': '#c1121f'}, color='#9d0208')

ax1.set_title('Memory escalation, thresholds, and restart loop evidence')
ax1.set_xlabel('Timestamp (UTC)')
ax1.set_ylabel('Memory (GB)')
ax2.set_ylabel('Restart count')
fmt_time_axis(ax1)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', ncols=2)
fig.tight_layout()
fig.savefig(ART / 'chart_03_memory_annotated.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Chart 7: milestone timeline, presentation-ready
milestones = [
    ('06:30', pd.Timestamp('2026-06-01T06:30:19Z'), 'GC warning', '#ffb703', 0.34),
    ('06:32', pd.Timestamp('2026-06-01T06:32:33Z'), 'Cache eviction failed', '#fb8500', 0.28),
    ('09:01', pd.Timestamp('2026-06-01T09:01:00Z'), 'Memory trend shift', '#ffd166', 0.34),
    ('18:22', pd.Timestamp('2026-06-01T18:22:00Z'), 'Memory > 60%', '#f4a261', 0.34),
    ('19:59', pd.Timestamp('2026-06-01T19:59:02Z'), 'OOMKill #1', '#e63946', 0.34),
    ('21:14-23:43', pd.Timestamp('2026-06-01T22:20:00Z'), 'Restart loop #2-#7', '#d00000', 0.28),
    ('23:04', ALERT_TIME, 'Alert fired', '#6c757d', 0.34),
]

fig, ax = plt.subplots(figsize=(16, 4.8))
ax.hlines(0, pd.Timestamp('2026-06-01T00:00:00Z'), pd.Timestamp('2026-06-02T00:00:00Z'), color='#adb5bd', linewidth=2)
for label_time, ts, label, color, y_text in milestones:
    ax.scatter(ts, 0, s=210, color=color, edgecolor='white', linewidth=1.4, zorder=3)
    ax.vlines(ts, 0.02, y_text - 0.05, color=color, linewidth=1.2, alpha=0.9)
    ax.text(ts, y_text, f'{label_time}\n{label}', ha='center', va='bottom', fontsize=9.5, color='#343a40')
ax.set_ylim(-0.12, 0.52)
ax.set_yticks([])
ax.set_title('ShopX incident timeline - from silent signal to alert')
ax.set_xlabel('Timestamp (UTC)')
fmt_time_axis(ax)
for spine in ['left', 'right', 'top']:
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(ART / 'chart_07_incident_timeline.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Optional support chart kept for appendix
fig, ax = plt.subplots(figsize=(14, 6))
top = cart_top20.head(12)
ax.barh(top['template'][::-1], top['count'][::-1], color='#4c78a8')
ax.set_title('Top 12 cart-service log templates')
ax.set_xlabel('Count')
ax.set_ylabel('Template')
fig.tight_layout()
fig.savefig(ART / 'log_templates_top12.png', dpi=160, bbox_inches='tight')
plt.show()


## Ket luan chinh

- Notebook logs nay da sua lai de phu hop voi bai nop chinh thuc.
- `GC overhead limit warning` la log signal som nhat trong du lieu thuc.
- `ProductCatalogCache eviction failed` la evidence root-cause manh nhat vi lap lai rat nhieu.
- Cac chart logs da duoc chuan hoa de dua thang vao report hoac slide.
